# 线性回归从零实现

## 导包

In [13]:
import random
import torch
print(torch.__version__)

1.12.0+cu113


## 构造数据集
使用线性模型参数$\mathbf{w} = [2, -3.4]^\top$、$b = 4.2$
和噪声项$\epsilon$生成数据集及其标签:
$$\mathbf{y}= \mathbf{X} \mathbf{w} + b + \mathbf\epsilon.$$

In [ ]:
def synthetic_data(w, b, num_examples):  
    """生成y=Xw+b+噪声"""
    X = torch.normal(0, 1, (num_examples,len(w))) #X的列和w的长度相同
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 1, y.size()) #噪声
    return X, y.reshape((-1, 1)) #返回x和y,这里y是二维的

In [15]:
#测试
test1 = torch.normal(0, 1, (1,3))
test1

tensor([[0.6707, 0.8659, 0.3875]])

形状验证

In [16]:
num_examples = 100
w = torch.tensor([2,-3.4], dtype=torch.float32)  
b = torch.tensor(4.2, dtype=torch.float32)

features, labels = synthetic_data(w, b, num_examples)  #w被广播为(1,3)然后与X相乘
print(f"feature[0]: {features[0]}", features.shape, "\n", f"labels[0]: {labels[0]}", labels.shape)


feature[0]: tensor([-0.4460,  0.9583]) torch.Size([100, 2]) 
 labels[0]: tensor([1.6665]) torch.Size([100, 1])


## 读取数据集
该函数能打乱数据集中的样本并以小批量方式获取数据




In [ ]:
def data_iter(batch_size, features, labels):
    num_features = len(features)
    index_features = list(range(num_features))
    # 打乱样本，随机读取
    random.shuffle(index_features)
    #小批次读取
    for i in range(0, num_features, batch_size):
        batch_features = torch.tensor(index_features[i:min(i + batch_size, num_features)]) #索引不要搞错
        yield features[batch_features], labels[batch_features]



采用小批量，因此函数执行一次就采样一小批样本

In [18]:
batch_size = 10

for X, y in data_iter(batch_size, features, labels):
    print(X, '\n', y)
    break #取第一个batch

    

tensor([[-1.4989,  0.2606],
        [-0.3578,  0.2951],
        [-0.7189, -1.4393],
        [-0.4496,  1.2546],
        [ 0.5767, -0.0048],
        [ 0.1840,  1.6811],
        [-0.8983, -1.2770],
        [ 0.4503,  0.0499],
        [-0.6772, -1.5782],
        [-1.5112,  0.1638]]) 
 tensor([[-0.4186],
        [ 2.6151],
        [ 7.2723],
        [-2.4062],
        [ 4.8244],
        [-1.8038],
        [ 7.0701],
        [ 5.4528],
        [ 8.7335],
        [-0.2744]])


## 初始化模型参数，定义模型、损失函数以及优化算法

通过从均值为0、标准差为0.01的正态分布中采样随机数来初始化权重，
并将偏置初始化为0。

优化算法是sgd：小批量随机梯度下降更新参数

In [ ]:
#初始化模型参数
W = torch.normal(0, 0.01, size=(2,1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

#模型
def linreg(X, W, b):
    return torch.matmul(X, W) + b

#损失函数
def squared_loss(y_hat, y):
    loss = (y_hat - y.reshape(y_hat.shape)) **2 / 2   #把数据集的标签y reshape成y_hat的形状，是一份保险
    return loss

#优化算法
def sgd(params, lr, batch_size):
    with torch.no_grad():  #更新参数值时要禁用梯度计算
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()  #梯度清零

## 训练
先算loss、再反向传播求梯度，最后用sgd更新参数，并打印epoch和loss

In [ ]:
lr = 0.1
batch_size = 10

for epoch in range(5):
    for X, y in data_iter(batch_size, features, labels):
        y_hat = linreg(X, W, b)
        loss = squared_loss(y_hat, y)   #loss是二维，形状是[batch,1]
        loss.sum().backward()   #必须sum成标量才能进行反向传播
        sgd([W,b], lr, batch_size) # 使用参数的梯度更新参数
    with torch.no_grad(): 
        train_loss = squared_loss(linreg(features, W, b), labels)  #注意这里用的是数据集的features和labels，唯一变的是W
        print(f'epoch: {epoch} ', f'train_loss: {train_loss.mean()}')  #这里用mean打印平均损失，不然放不下

epoch: 0  train_loss: 2.737973213195801
epoch: 1  train_loss: 0.7426214814186096
epoch: 2  train_loss: 0.44431576132774353
epoch: 3  train_loss: 0.40339308977127075
epoch: 4  train_loss: 0.3943616449832916
